# 🔍 Task 2: Exploratory Data Analysis (EDA)
### CodeAlpha Data Analytics Internship
**Dataset:** Titanic — Survival Prediction Dataset  
**Source:** Built-in via seaborn (`sns.load_dataset('titanic')`) — no download needed!  
**Goal:** Uncover patterns behind passenger survival using statistical exploration and visualization.

In [ ]:
# ─────────────────────────────────────────────
# STEP 1: Import Libraries & Load Dataset
# ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Load Titanic dataset (built into seaborn — no file needed!)
df = sns.load_dataset('titanic')

print("✅ Dataset loaded!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

In [ ]:
# ─────────────────────────────────────────────
# STEP 2: Data Structure Overview
# ─────────────────────────────────────────────

print("=" * 50)
print("📋 COLUMN NAMES & DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("❓ MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "=" * 50)
print("📊 BASIC STATISTICS (Numeric)")
print("=" * 50)
df.describe()

In [ ]:
# ─────────────────────────────────────────────
# STEP 3: Missing Value Heatmap
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Missing Value Analysis', fontsize=16, fontweight='bold')

# Heatmap
sns.heatmap(df.isnull(), cbar=True, cmap='YlOrRd', 
            yticklabels=False, ax=axes[0])
axes[0].set_title('Missing Value Heatmap', fontsize=13)

# Bar chart of missing %
missing_cols = df.isnull().sum()[df.isnull().sum() > 0]
colors = ['#e74c3c' if v > 50 else '#f39c12' for v in missing_cols.values]
axes[1].barh(missing_cols.index, missing_cols.values, color=colors, edgecolor='white')
axes[1].set_title('Missing Value Count per Column', fontsize=13)
axes[1].set_xlabel('Missing Count')
for i, v in enumerate(missing_cols.values):
    axes[1].text(v+1, i, str(v), va='center', fontsize=11)

plt.tight_layout()
plt.savefig('Task2_EDA/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: missing_values.png")

In [ ]:
# ─────────────────────────────────────────────
# STEP 4: Survival Rate Analysis (Core EDA)
# ─────────────────────────────────────────────

fig = plt.figure(figsize=(20, 12))
fig.suptitle('🚢 Titanic Survival Analysis', fontsize=22, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Overall Survival
ax1 = fig.add_subplot(gs[0, 0])
survival_counts = df['survived'].value_counts()
colors = ['#e74c3c', '#2ecc71']
wedges, texts, autotexts = ax1.pie(survival_counts, labels=['Did Not Survive','Survived'],
        autopct='%1.1f%%', colors=colors, startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':2})
for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')
ax1.set_title('Overall Survival Rate', fontsize=13, fontweight='bold')

# 2. Survival by Gender
ax2 = fig.add_subplot(gs[0, 1])
gender_survival = df.groupby('sex')['survived'].mean() * 100
bars = ax2.bar(gender_survival.index, gender_survival.values,
               color=['#3498db','#e91e8c'], edgecolor='white', linewidth=1.5, width=0.5)
ax2.set_title('Survival Rate by Gender', fontsize=13, fontweight='bold')
ax2.set_ylabel('Survival Rate (%)')
ax2.set_ylim(0, 100)
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
             f'{bar.get_height():.1f}%', ha='center', fontsize=12, fontweight='bold')

# 3. Survival by Class
ax3 = fig.add_subplot(gs[0, 2])
sns.countplot(data=df, x='pclass', hue='survived', 
              palette={0:'#e74c3c', 1:'#2ecc71'}, ax=ax3, edgecolor='white')
ax3.set_title('Survival by Passenger Class', fontsize=13, fontweight='bold')
ax3.set_xlabel('Passenger Class (1=First, 3=Third)')
ax3.legend(['Did Not Survive','Survived'])

# 4. Age Distribution by Survival
ax4 = fig.add_subplot(gs[1, 0])
df[df['survived']==1]['age'].dropna().plot(kind='hist', bins=25, alpha=0.7, 
    color='#2ecc71', label='Survived', ax=ax4, edgecolor='white')
df[df['survived']==0]['age'].dropna().plot(kind='hist', bins=25, alpha=0.7, 
    color='#e74c3c', label='Did Not Survive', ax=ax4, edgecolor='white')
ax4.set_title('Age Distribution by Survival', fontsize=13, fontweight='bold')
ax4.set_xlabel('Age')
ax4.legend()

# 5. Fare vs Survival (Boxplot)
ax5 = fig.add_subplot(gs[1, 1])
df.boxplot(column='fare', by='survived', ax=ax5,
           boxprops=dict(color='#2c3e50'),
           medianprops=dict(color='red', linewidth=2))
ax5.set_title('Fare Distribution by Survival', fontsize=13, fontweight='bold')
ax5.set_xlabel('Survived (0=No, 1=Yes)')
ax5.set_ylabel('Fare (£)')
plt.sca(ax5)
plt.title('')

# 6. Survival by Embarkation Port
ax6 = fig.add_subplot(gs[1, 2])
emb_survival = df.groupby('embark_town')['survived'].mean() * 100
bars = ax6.bar(emb_survival.index, emb_survival.values,
               color=['#9b59b6','#3498db','#1abc9c'], edgecolor='white', linewidth=1.5)
ax6.set_title('Survival Rate by Embarkation Port', fontsize=13, fontweight='bold')
ax6.set_ylabel('Survival Rate (%)')
ax6.set_ylim(0, 80)
for bar in bars:
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{bar.get_height():.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('Task2_EDA/survival_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: survival_analysis.png")

In [ ]:
# ─────────────────────────────────────────────
# STEP 5: Correlation Heatmap
# ─────────────────────────────────────────────

plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))  # show only lower triangle
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, square=True, linewidths=0.5,
            annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Correlation Matrix — Titanic Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('Task2_EDA/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: correlation_matrix.png")

In [ ]:
# ─────────────────────────────────────────────
# STEP 6: Statistical Summary & Hypotheses
# ─────────────────────────────────────────────

print("=" * 55)
print("📊 KEY STATISTICAL FINDINGS")
print("=" * 55)

total = len(df)
survived = df['survived'].sum()
print(f"Total Passengers    : {total}")
print(f"Survived            : {survived} ({survived/total*100:.1f}%)")
print(f"Did Not Survive     : {total-survived} ({(total-survived)/total*100:.1f}%)")

print("\n── By Gender ──────────────────────────────")
for sex in ['female','male']:
    g = df[df['sex']==sex]
    sr = g['survived'].mean()*100
    print(f"  {sex.capitalize():8}: {sr:.1f}% survival rate  (n={len(g)})")

print("\n── By Class ───────────────────────────────")
for cls in [1,2,3]:
    g = df[df['pclass']==cls]
    sr = g['survived'].mean()*100
    print(f"  Class {cls}: {sr:.1f}% survival rate  (n={len(g)})")

print("\n── By Age Group ───────────────────────────")
df['age_group'] = pd.cut(df['age'], bins=[0,12,18,35,60,100],
                          labels=['Child','Teen','Young Adult','Adult','Senior'])
for grp in ['Child','Teen','Young Adult','Adult','Senior']:
    g = df[df['age_group']==grp]
    sr = g['survived'].mean()*100
    print(f"  {grp:15}: {sr:.1f}% survival rate  (n={len(g)})")

## ✅ Task 2 Key Findings

| Question | Finding |
|----------|---------|
| Overall survival rate | ~38% survived |
| Gender impact | Females: ~74% survived vs Males: ~19% |
| Class impact | 1st class: ~63% vs 3rd class: ~24% |
| Age impact | Children had highest survival rates |
| Fare correlation | Higher fare → higher survival chance |

**Conclusion:** Gender, class, and age were the strongest predictors of survival — matching the "women and children first" evacuation protocol.